In [19]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)


In [2]:
orders = pd.read_csv(r'A:\Olist Project\Dataset\olist_orders_dataset.csv')
items = pd.read_csv(r'A:\Olist Project\Dataset\olist_order_items_dataset.csv')
customers = pd.read_csv(r'A:\Olist Project\Dataset\olist_customers_dataset.csv')
sellers = pd.read_csv(r'A:\Olist Project\Dataset\olist_sellers_dataset.csv')
products = pd.read_csv(r'A:\Olist Project\Dataset\olist_products_dataset.csv')
reviews = pd.read_csv(r'A:\Olist Project\Dataset\olist_order_reviews_dataset.csv')
payments = pd.read_csv(r'A:\Olist Project\Dataset\olist_order_payments_dataset.csv')
categories = pd.read_csv(r'A:\Olist Project\Dataset\product_category_name_translation.csv')

## Exploration of Dataset

In [20]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


In [21]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

We need to fix data type of date-time columns and filter out only those orders that has been delivered, because we can only analyze the impact of delivery on ratings only on orders that have been delivered.

### fixing data type

In [22]:
date_columns = ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date',
'order_delivered_customer_date','order_estimated_delivery_date']

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

### Filtering to delivered only

In [23]:
print(orders['order_status'].value_counts())

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [24]:
delivered = orders[orders['order_status']=='delivered'].copy()

In [25]:
delivered['order_status'].value_counts()

order_status
delivered    96478
Name: count, dtype: int64

In [29]:
delivered.isnull().sum()

order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64

In [30]:
to_drop = ['order_approved_at','order_delivered_carrier_date','order_delivered_customer_date']
delivered.dropna(subset = to_drop, inplace = True)

In [31]:
delivered.isnull().sum()

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
dtype: int64

We are done with preparation of our order dataset and we will now inspect items dataset to see if we require any changes

In [26]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [27]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [32]:
items.isnull().sum()

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

In [33]:
items['shipping_limit_date'] =  pd.to_datetime(items['shipping_limit_date'])

In [34]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


items table is now ready for use. We move to reviews table.

In [35]:
reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [36]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [37]:
reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

in reviews table we are only concerned with order_id and reviews_score, both are non-null and of correct type. We can move to products table.

In [39]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,1000.00,30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00


In [40]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


In [41]:
products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Products table has only column that is valuable to us i.e. product ID and it is non-null and correct data type. We can move to customers table.

In [42]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [43]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB


In [44]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Customer table is ready for use. We can move to next table i.e. sellers.

In [45]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [46]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [47]:
sellers.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

sellers table is ready to use. We can move to next table i.e. categories.

In [49]:
categories.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [51]:
categories.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [52]:
categories.isnull().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

Categories table is ready for use. We can move to next table i.e. payments.

In [53]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


Payments table has no column that we can use in our delivery analysis. So we can now move to our Metrics.

## Core Metrics

### Delivery Delay

In [54]:
delivered['delivery_delay'] = (delivered['order_delivered_customer_date'] -  delivered['order_estimated_delivery_date']).dt.days

In [58]:
delivered['delivery_delay'].head()

0    -8
1    -6
2   -18
3   -13
4   -10
Name: delivery_delay, dtype: int64

We have -ve, 0 and +ve values in our delivery_delay column. -ve = early delivery    0 =  on time delivery   +ve means = late delivery. Now we will create a column to see if the order is delayed or not.

In [60]:
delivered['is_late'] = delivered['delivery_delay'] >0

In [61]:
delivered['is_late'].head()

0    False
1    False
2    False
3    False
4    False
Name: is_late, dtype: bool

Now we have info about the number of delays and if the order was delayed or not.

### Actual delivery days

In [62]:
delivered.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10,False


In [63]:
delivered['actual_delivery_days'] = (delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']).dt.days

In [64]:
delivered['actual_delivery_days'].head()

0     8
1    13
2     9
3    13
4     2
Name: actual_delivery_days, dtype: int64

We have calculated total delivery days.

### Sellers Vs logistics split in delivery

In [66]:
delivered['seller_days'] = (delivered['order_delivered_carrier_date'] - delivered['order_purchase_timestamp']).dt.days
#time taken by sellers to hand order to logistics partners

In [67]:
delivered['logistic_days'] = (delivered['order_delivered_customer_date']-delivered['order_delivered_carrier_date']).dt.days
#time take by logistics to readh to customers

In [68]:
print(f'Avg seller processing: {delivered["seller_days"].mean():.1f} days')
print(f'Avg logistics time:    {delivered["logistic_days"].mean():.1f} days')
print(f'Avg total delivery:    {delivered["actual_delivery_days"].mean():.1f} days')


Avg seller processing: 2.7 days
Avg logistics time:    8.9 days
Avg total delivery:    12.1 days


### Now we can merge our tables into one master dataset for further analysis

In [72]:
delivered.shape

(96455, 13)

In [69]:
#1. Merginf with customers customers with delivered table as our base
df = delivered.merge(customers[['customer_id', 'customer_state']], on ='customer_id', how ='left')

In [71]:
df.shape

(96455, 14)

In [73]:
#2. Merging with items table with df as base
df = df.merge(items[['order_id','product_id','seller_id','price']], on ='order_id',how='left')

In [74]:
df.shape

(110173, 17)

No of rows have increased here because items table has multiple entries for 1 order ID

In [77]:
#3. Merging products with category Transaltion
products = products.merge(categories, on ='product_category_name', how='left')

In [78]:
#4. Add translated categories to main df
df = df.merge(products[['product_id','product_category_name_english']], on ='product_id', how ='left')

In [79]:
df.shape

(110173, 18)

In [81]:
#5. Add review scores
df = df.merge(reviews[['order_id','review_score']], on='order_id',how='left')

In [83]:
df.shape

(110816, 19)

In [82]:
#6. Clean up column name
df.rename(columns={'product_category_name_english': 'category'}, inplace=True)

In [88]:
df=df.drop_duplicates(subset='order_id')

Note: Each order retains the first matched item's category after deduplication. Multi-item orders spanning multiple categories are assigned one category only. Category-level findings should be interpreted directionally.

In [89]:
df.shape

(96455, 19)

In [90]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date        0
order_delivered_customer_date       0
order_estimated_delivery_date       0
delivery_delay                      0
is_late                             0
actual_delivery_days                0
seller_days                         0
logistic_days                       0
customer_state                      0
product_id                          0
seller_id                           0
price                               0
category                         1377
review_score                      646
dtype: int64

In [91]:
df = df.dropna(subset = 'review_score')

In [92]:
df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date        0
order_delivered_customer_date       0
order_estimated_delivery_date       0
delivery_delay                      0
is_late                             0
actual_delivery_days                0
seller_days                         0
logistic_days                       0
customer_state                      0
product_id                          0
seller_id                           0
price                               0
category                         1368
review_score                        0
dtype: int64

We have removed rows that do not have review score because reviews is are metric to analyze delivery performance.

### Exploratory Analysis 

#### Overall Late delivery rate

In [96]:
late_rate = df['is_late'].mean()*100
print(late_rate)

6.659082132158773


In [98]:
avg_delay_when_late = df[df['is_late']]['delivery_delay'].mean()
print(avg_delay_when_late)

10.519122257053292


In [99]:
total_late = df['is_late'].sum()
print(total_late)

6380


In [102]:
print(f'Late delivery rate:{late_rate:.1f}%')
print(f'Total late orders:{total_late:,}')
print(f'Avg delay when late:{avg_delay_when_late:.1f} days')
print(f'Median delay (all orders):{df["delivery_delay"].median():.1f} days')
print(f'Max delay:{df["delivery_delay"].max():.0f} days')


Late delivery rate:6.7%
Total late orders:6,380
Avg delay when late:10.5 days
Median delay (all orders):-12.0 days
Max delay:188 days


#### Sellers vs Logistics Overall

In [106]:
print(f'Avg seller processing:{df["seller_days"].mean():.1f} days')
print(f'Avg logistics time:{df["logistic_days"].mean():.1f} days')
print(f'Avg total delivery:{df["actual_delivery_days"].mean():.1f} days')


Avg seller processing:2.7 days
Avg logistics time:8.8 days
Avg total delivery:12.1 days


In [108]:
# What % of total delivery time is logistics?
pct = df['logistic_days'].mean() / df['actual_delivery_days'].mean() * 100
print(f'Logistics as % of total delivery: {pct:.0f}%')


Logistics as % of total delivery: 73%


#### Delivery by state

In [111]:
state_delay = df.groupby('customer_state').agg(total_orders=('order_id','count'),
                                              late_orders = ('is_late','sum'),
                                              avg_delay_when_late = ('delivery_delay', lambda x: round(x[x > 0].mean(), 1)),
                                              avg_review   = ('review_score', 'mean')).reset_index()


In [112]:
state_delay['delay_rate'] = (state_delay['late_orders']/state_delay['total_orders']*100).round(1)

In [114]:
state_delay['avg_review'] = state_delay['avg_review'].round(2)

In [119]:
state_delay = state_delay[['customer_state', 'total_orders', 'late_orders','delay_rate', 'avg_delay_when_late', 'avg_review']]
state_delay = state_delay.sort_values('delay_rate', ascending=False)
state_delay.head()

,customer_state,total_orders,late_orders,delay_rate,avg_delay_when_late,avg_review
1,AL,394,82,20.80,9.70,3.86
9,MA,711,122,17.20,10.50,3.83
24,SE,334,50,15.00,16.40,3.91
16,PI,471,65,13.80,13.50,3.99
5,CE,1272,174,13.70,15.00,3.94


#### Delay by Product Category

In [121]:
cat_delay = df.groupby('category').agg(total_orders = ('order_id', 'count'),
                                       late_orders  = ('is_late', 'sum'),
                                       avg_delay_when_late = ('delivery_delay',
                                       lambda x: round(x[x > 0].mean(), 1)),
                                       avg_review   = ('review_score', 'mean')).reset_index()

cat_delay['late_rate'] = (cat_delay['late_orders'] / cat_delay['total_orders'] * 100).round(1)
cat_delay['avg_review'] = cat_delay['avg_review'].round(2)

cat_delay = cat_delay[['category', 'total_orders', 'late_orders','late_rate', 'avg_delay_when_late', 'avg_review']]
cat_delay = cat_delay.sort_values('late_rate', ascending=False)

cat_delay.head(15)


,category,total_orders,late_orders,late_rate,avg_delay_when_late,avg_review
46,home_comfort_2,21,3,14.30,12.70,3.90
41,furniture_mattress_and_upholstery,37,5,13.50,15.20,3.89
4,audio,341,40,11.70,8.40,3.84
47,home_confort,368,35,9.50,13.60,3.92
33,fashion_underwear_beach,116,11,9.50,8.40,4.01
10,books_technical,254,21,8.30,6.50,4.43
6,baby,2741,220,8.00,11.10,4.12
57,office_furniture,1236,99,8.00,13.00,3.65
56,musical_instruments,601,46,7.70,12.40,4.25
26,electronics,2488,190,7.60,9.40,4.13


#### Seller vs Logistics Split by Category

In [122]:
processing_by_cat = df.groupby('category').agg(total_orders = ('order_id', 'count'),
                                               avg_seller_processing = ('seller_days', 'mean'),
                                               avg_logistics = ('logistic_days', 'mean'),
                                               late_rate = ('is_late', 'mean')).reset_index()

processing_by_cat['late_rate'] = (processing_by_cat['late_rate'] * 100).round(1)
processing_by_cat['avg_seller_processing'] = (processing_by_cat['avg_seller_processing'].round(1))
processing_by_cat['avg_logistics'] = (processing_by_cat['avg_logistics'].round(1))

processing_by_cat = processing_by_cat.sort_values('avg_seller_processing', ascending=False)

processing_by_cat.head(10)


,category,total_orders,avg_seller_processing,avg_logistics,late_rate
57,office_furniture,1236,9.90,9.70,8.00
31,fashion_shoes,229,5.10,9.60,4.40
27,fashio_female_clothing,36,4.60,7.30,5.60
38,furniture_bedroom,89,4.20,8.80,6.70
45,home_appliances_2,225,4.10,8.80,6.20
30,fashion_male_clothing,105,4.10,7.60,3.80
48,home_construction,464,4.00,8.60,6.20
33,fashion_underwear_beach,116,3.80,9.30,9.50
23,diapers_and_hygiene,25,3.70,5.90,0.00
2,art,189,3.50,6.90,6.30


#### Delay Buckets and Review Score Relationship

In [125]:
df['delay_bucket'] = pd.cut(df['delivery_delay'],bins=[-999, -1, 0, 3, 7, 999], 
                            labels=['Early', 'On Time', '1-3 days late','4-7 days late', '7+ days late'])

review_by_delay = df.groupby('delay_bucket')['review_score'].mean().round(2)
print(review_by_delay)

# % of 1-star reviews per bucket 
one_star = df.groupby('delay_bucket').apply(lambda x: (x['review_score'] == 1).mean() * 100).round(1)
print('\n% of 1-star reviews:')
print(one_star)


delay_bucket
Early           4.29
On Time         4.03
1-3 days late   3.29
4-7 days late   2.10
7+ days late    1.70
Name: review_score, dtype: float64

% of 1-star reviews:
delay_bucket
Early            6.60
On Time          8.60
1-3 days late   25.10
4-7 days late   58.50
7+ days late    69.80
dtype: float64


Early: 4.29 stars | On Time: 4.03 | 1-3 days late: 3.29 | 4-7 days: 2.10 | 7+ days: 1.70

The sharpest drop is from On Time to 1-3 days late - a 0.74 point fall and from 1-3 days late to 4-7 days late - a 1.19, his means preventing lateness is valuable.

Almost 70% of 7+ days late orders had 1 star rating which is a strong indication that delivery time influence ratings.


#### Correlation Analysis

In [126]:
# Full correlation i.e. all orders including early and on-time
corr_full = df['delivery_delay'].corr(df['review_score'])

# Late orders only i.e. how severity of delay drives ratings down
late_only = df[df['is_late']]
corr_late = late_only['delivery_delay'].corr(late_only['review_score'])

print(f'Correlation (all orders): {corr_full:.3f}')
print(f'Correlation (late only):  {corr_late:.3f}')


Correlation (all orders): -0.267
Correlation (late only):  -0.147


corr_full (-0.267): Moderate negative i.e. delivery timing meaningfully influences satisfaction across the full spectrum

corr_late (-0.147): Weaker than expected i.e. satisfaction damage concentrates at the MOMENT of lateness, not progressively with each extra day

Business implication: Preventing any lateness is more valuable than reducing delay duration once late


#### Monthly Trend Analysis

In [130]:
df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')

monthly = df.groupby('order_month').agg(orders = ('order_id', 'count'), late_rate= ('is_late', 'mean')).reset_index()

monthly['late_rate'] = (monthly['late_rate'] * 100).round(1)
monthly['order_month_str'] = monthly['order_month'].astype(str)

# Show last 12 months
print(monthly.tail(12).to_string(index=False))


order_month  orders  late_rate order_month_str
    2017-09    4117       4.30         2017-09
    2017-10    4446       4.00         2017-10
    2017-11    7237      12.30         2017-11
    2017-12    5461       7.30         2017-12
    2018-01    7013       5.60         2018-01
    2018-02    6507      14.00         2018-02
    2018-03    6948      18.70         2018-03
    2018-04    6752       4.40         2018-04
    2018-05    6722       6.50         2018-05
    2018-06    6072       1.20         2018-06
    2018-07    6118       3.30         2018-07
    2018-08    6330       6.10         2018-08


While no consistent long-term trend is visible, a notable spike occurs in Feb-Mar 2018 (peaking at 18.7%) before recovering sharply in April 2018.
This anomaly is not random noise, it is something warrants investigation into operational changes, carrier changes, or volume surge during that period.


## Exporting the csv file

In [133]:
# 1. Master file
df[['order_id', 'customer_state', 'category', 'delivery_delay', 'is_late', 'seller_days', 'logistic_days',
    'review_score', 'order_purchase_timestamp', 'delay_bucket']].to_csv('olist_master.csv', index=False)

# 2. State-level aggregation
state_delay.to_csv('state_performance.csv', index=False)

# 3. Category-level aggregation
cat_delay.to_csv('category_delay.csv', index=False)

# 4. Monthly trend
monthly.to_csv('monthly_trend.csv', index=False)

# 5. Seller vs logistics by category
processing_by_cat.to_csv('processing_by_cat.csv', index=False)
